# 11 – Ratings

Esplorazione e data cleaning del dataset `ratings.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `username` | Nome utente MAL |
| `anime_id` | ID dell'anime su MAL |
| `status` | Stato di visione |
| `score` | Voto dell'utente |
| `is_rewatching` | Indica se l'utente sta rivedendo l'anime |
| `num_watched_episodes` | Numero di episodi guardati |

## 1. Import e caricamento dati

Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [47]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_rt = pd.read_csv('../datasets/ratings.csv')
print(f'Shape: {df_rt.shape}')
df_rt.info(show_counts=True)
df_rt.head()

Shape: (124298357, 6)
<class 'pandas.DataFrame'>
RangeIndex: 124298357 entries, 0 to 124298356
Data columns (total 6 columns):
 #   Column                Non-Null Count      Dtype  
---  ------                --------------      -----  
 0   username              124298350 non-null  str    
 1   anime_id              124298357 non-null  int64  
 2   status                124298357 non-null  str    
 3   score                 124298357 non-null  int64  
 4   is_rewatching         120501036 non-null  float64
 5   num_watched_episodes  124298357 non-null  int64  
dtypes: float64(1), int64(3), str(2)
memory usage: 5.6 GB


,username,anime_id,status,score,is_rewatching,num_watched_episodes
0,--------788,30276,watching,7,0.0,3
1,--------788,28851,completed,7,0.0,1
2,--------788,41168,completed,7,0.0,1
3,--------788,22199,completed,10,0.0,24
4,--------788,16498,completed,10,0.0,25


**Osservazioni iniziali:**
- Il dataset contiene 124,298,357 di righe e 6 colonne.
- `username` presenta 7 righe nulle.
- `is_rewatching` contiene 3,797,321 valori null che controlliamo succesivamente per verificare se si tratta di un errore o meno. 

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

La dimensione del dataset (~124M righe) rende la verifica tramite `df.duplicated()` proibitiva in termini di memoria. Usiamo `pd.util.hash_pandas_object()` che calcola un hash `uint64` per ogni riga e cerca duplicati su quella serie, evitando di confrontare le righe intere.

In [48]:
n_originale = len(df_rt)

hashes = pd.util.hash_pandas_object(df_rt, index=False)
n_dup = hashes.duplicated(keep=False).sum()
print(f'Duplicati esatti sull\'intero dataset: {n_dup:,}')
if n_dup == 0:
    print('→ Nessun duplicato esatto, nessuna operazione richiesta.')
else:
    print('→ Presenza di duplicati: rimozione necessaria.')
    df_rt = df_rt[~hashes.duplicated(keep='first')].reset_index(drop=True)
    print(f'Righe dopo rimozione: {len(df_rt):,}')

Duplicati esatti sull'intero dataset: 0
→ Nessun duplicato esatto, nessuna operazione richiesta.


Nessun duplicato esatto trovato. Tutte le righe sono già uniche. Il dataset rimane invariato.

Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `username`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria `username` di `profiles.csv`.

I valori duplicati sono **attesi**: lo stesso utente ha più righe (una per ogni anime in lista).

I controlli rilevanti sono:
- **Valori nulli**: una riga senza username non è collegabile a nessun profilo e va rimossa.
- **Integrità referenziale**: ogni username deve esistere in `profiles_clean.csv`.

Usiamo `check_fk` al posto di `analyze`.

In [49]:
df_rt['username'] = df_rt['username'].str.strip()
df_profiles = pd.read_csv('../datasets_cleaned/profiles_clean.csv', usecols=['username'])

mask_orphan_user = check_fk(df_rt['username'], df_profiles['username'], child_df=df_rt)

print(f'Null in username               : {df_rt["username"].isna().sum()}')
print(f'Duplicati in username (attesi) : {df_rt["username"].duplicated().sum():,}')


  Colonna FK  (tabella figlia)        username
  Colonna PK  (tabella padre)         username
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       124,298,357
  Valori null nella FK                7  (0.00%)
  Valori non-null nella FK            124,298,350  (100.00%)
  Valori unici nella PK padre         335,476

  ✓  Righe con FK valida              123,658,186  (99.48%)
  ✗  Righe orfane (FK non in PK)      640,164  (0.52%)
     → ID orfani unici                1,673

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

640,164 riga/e (0.52%) violano l'integrità referenziale.

Campione righe orfane (prime 10)
────────────────────────────────────────────────────────────────────────────────

     username  anime_id     status  score  is_rewatching  num_wat

**Osservazioni:**

- Sono presenti 7 righe con `username` null. Senza identificatore utente la riga non è collegabile a nessun profilo quindi va rimossa. 
- Ci sono 640,164 righe orfane che vanno rimosse. 

In [50]:
#Rimozione delle righe nulle
print(f'Righe con username null: {df_rt["username"].isna().sum()}')
print()

df_rt.dropna(subset=['username'], inplace=True)
print(f'\nRighe dopo rimozione: {len(df_rt):,}')

#Rimozione delle righe orfane
if mask_orphan_user.any():
    n_orfane = mask_orphan_user.sum()
    df_rt = df_rt[~mask_orphan_user].reset_index(drop=True)
    print(f'Righe orfane rimosse : {n_orfane:,}')
    print(f'Righe rimanenti      : {len(df_rt):,}')
else:
    print('Nessuna riga orfana da rimuovere.')

Righe con username null: 7


Righe dopo rimozione: 124,298,350
Righe orfane rimosse : 640,164
Righe rimanenti      : 123,658,186


### 2.2 `anime_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria `mal_id` di `details.csv`.

I valori duplicati sono **attesi**: lo stesso anime compare in più righe (uno per utente).

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID deve esistere in `details_clean.csv`.

Usiamo `check_fk` al posto di `analyze`.

In [51]:
df_details = pd.read_csv('../datasets_cleaned/details_clean.csv', usecols=['mal_id'])

mask_orphan_anime = check_fk(df_rt['anime_id'], df_details['mal_id'], child_df=df_rt)

print(f'Null in anime_id               : {df_rt["anime_id"].isna().sum()}')
print(f'Duplicati in anime_id (attesi) : {df_rt["anime_id"].duplicated().sum():,}')


  Colonna FK  (tabella figlia)        anime_id
  Colonna PK  (tabella padre)         mal_id
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       123,658,186
  Valori null nella FK                0  (0.00%)
  Valori non-null nella FK            123,658,186  (100.00%)
  Valori unici nella PK padre         28,955

  ✓  Righe con FK valida              123,351,961  (99.75%)
  ✗  Righe orfane (FK non in PK)      306,225  (0.25%)
     → ID orfani unici                326

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

306,225 riga/e (0.25%) violano l'integrità referenziale.

Campione righe orfane (prime 10)
────────────────────────────────────────────────────────────────────────────────

            username  anime_id         status  score  is_rewatching  n

**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID anime valido.
- **Integrità referenziale**: ci sono 306,225 righe orfane che rimuoviamo. 

In [52]:
if mask_orphan_anime.any():
    n_orfane = mask_orphan_anime.sum()
    df_rt = df_rt[~mask_orphan_anime].reset_index(drop=True)
    print(f'Righe orfane rimosse : {n_orfane:,}')
    print(f'Righe rimanenti      : {len(df_rt):,}')
else:
    print('Nessuna riga orfana da rimuovere.')

Righe orfane rimosse : 306,225
Righe rimanenti      : 123,351,961


### 2.3 `status`

Questa colonna indica lo stato di visione dell'anime da parte dell'utente. Controlliamo l'eventuale presenza di valori nulli e anomali. Usiamo `analyze` per l'ispezione.

In [53]:
df_rt['status'] = df_rt['status'].str.strip()
analyze(df_rt['status'])


  Nome serie                     status
  dtype                          str
  Dimensione                     123,351,961
  Range indice                   0 … 123351960
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   123,351,961
  Valori non nulli               123,351,961  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   6  (0.00%)
  Valori duplicati               123,351,955  (100.00%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  7  caratteri
  Lunghezza max                  13  caratteri
  Lunghezza media                9.84  caratteri
  Lunghezza mediana              9.0000  caratteri
  Dev. standard lunghezza        1.91
  IQR lunghezza                  4.00

**Osservazioni:**

- Non ci sono righe nulle e i valori duplicati sono attesi. 
- 16.690 righe presentano `status = 'unknown'`. Stampiamo un campione per capire se va rimosso. 

In [54]:
print('Campione di 20 righe con status = "unknown":')
df_rt[df_rt['status'] == 'unknown'].sample(n=20, random_state=42)

Campione di 20 righe con status = "unknown":


,username,anime_id,status,score,is_rewatching,num_watched_episodes
21397498,brandon9565018,49470,unknown,0,0.0,0
7715386,CatzillaOP,30296,unknown,0,0.0,1
35257429,NightFury000,25731,unknown,0,0.0,1
270137,isthisgood123,39196,unknown,0,0.0,0
118437834,The_Audacity,42423,unknown,0,0.0,0
108191707,Rayquendil,37932,unknown,0,0.0,0
86269306,AylaTheIdiot,19685,unknown,0,0.0,1
88130125,AzuraelT,38435,unknown,0,0.0,1
109582061,Real_wozza,33558,unknown,0,0.0,1
87248954,namaime,53876,unknown,0,0.0,0


Si nota che le righe con status `unknown` sono righe che non contengono altri valori utili e vanno rimosse. 

In [55]:
mask_unknown = df_rt['status'] == 'unknown'
print(f'Righe con status=unknown : {mask_unknown.sum():,}')
df_rt = df_rt[~mask_unknown].copy()
print(f'\nRighe dopo rimozione: {len(df_rt):,}')

Righe con status=unknown : 16,656

Righe dopo rimozione: 123,335,305


### 2.4 `score`

Questa colonna contiene il voto assegnato dall'utente all'anime, su scala intera da 0 a 10. Controlliamo la presenza di valori nulli o anomali. Usiamo `analyze` per l'ispezione.

In [56]:
analyze(df_rt['score'])


  Nome serie                     score
  dtype                          int64
  Dimensione                     123,335,305
  Range indice                   0 … 123351960
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   123,335,305
  Valori non nulli               123,335,305  (100.00%)
  Null / NaN                     0  (0.00%)
  Zeri                           54,235,027  (43.97%)
  Positivi                       69,100,278   (56.03%)
  Negativi                       0   (0.00%)
  Valori unici                   11  (0.00%)

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            0
  Max                            10
  Range                          10
  Media                          4.0953
  Mediana                        5.0000
  Moda/e                         0
  Dev. standar

**Osservazioni:**  
- Nessun null, dtype già `int64`. 
- Ci sono 11 valori unici che corrispondono che le valutazioni da 0 a 10. 
- Il valore `0` indica che l'utente **non ha assegnato un voto** (~44% delle righe): è un comportamento atteso su MAL in quano un anime può essere in lista senza essere valutato.

Nessuna pulizia necessaria 

### 2.5 `is_rewatching`

Questa colonna indica se l'utente sta rivedendo l'anime per la seconda volta (o più). Usiamo `analyze` per l'ispezione.

In [57]:
analyze(df_rt['is_rewatching'])


  Nome serie                     is_rewatching
  dtype                          float64
  Dimensione                     123,335,305
  Range indice                   0 … 123351960
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   123,335,305
  Valori non nulli               119,560,277  (96.94%)
  Null / NaN                     3,775,028  (3.06%)
  Zeri                           119,457,481  (96.86%)
  Positivi                       102,796   (0.08%)
  Negativi                       0   (0.00%)
  Valori unici                   2  (0.00%)
  Valori float                   0

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            0.0000
  Max                            1.0000
  Range                          1.0000
  Media                          0.0009
  Mediana                   

**Osservazioni:**  

- Ci sono solo 2 valori unici: `1` e `0`, quindi si tratta di un flag booleano.
- Ci sono **3.775.028 valori null**. Verifichiamo se corrispondono a un `status` specifico come `watching`, `dropped`, `plan_to_watch`, etc.
- Dtype `float64` con valori `0.0`, `1.0` e `NaN` va convertito a `boolean`.

In [58]:
null_rew  = df_rt['is_rewatching'].isna()
false_rew = df_rt['is_rewatching'] == 0.0
true_rew  = df_rt['is_rewatching'] == 1.0

print(f'Righe is_rewatching = NaN   : {null_rew.sum():>10,}')
print(f'Righe is_rewatching = 0.0   : {false_rew.sum():>10,}')
print(f'Righe is_rewatching = 1.0   : {true_rew.sum():>10,}')
print()

print('Distribuzione status per is_rewatching = NaN:')
print(df_rt[null_rew]['status'].value_counts(dropna=False).to_string())
print()
print('Distribuzione status per is_rewatching = 0.0 (campione top 5):')
print(df_rt[false_rew]['status'].value_counts(dropna=False).head().to_string())

Righe is_rewatching = NaN   :  3,775,028
Righe is_rewatching = 0.0   : 119,457,481
Righe is_rewatching = 1.0   :    102,796

Distribuzione status per is_rewatching = NaN:
status
completed        3006630
plan_to_watch     315522
dropped           209242
on_hold           168857
watching           74777

Distribuzione status per is_rewatching = 0.0 (campione top 5):
status
completed        75468198
plan_to_watch    31040953
watching          5550231
dropped           4341081
on_hold           3057018


**Osservazioni sulla correlazione `is_rewatching` ↔ `status`:**

I null in `is_rewatching` non corrispondono a un singolo `status` ma sono distribuiti tra tutti i valori. Confermano che si tratta di un valore di default assegnato agli anime aggiunti alla lista prima che MAL introducesse il tracking esplicito del re-watching.

**Nessuna pulizia aggiuntiva necessaria**: i null sono strutturali e vengono mantenuti.Facciamo solo la conversione a `boolean`.

In [59]:
print('Valori is_rewatching prima della conversione:')
print(df_rt['is_rewatching'].value_counts(dropna=False))

df_rt['is_rewatching'] = df_rt['is_rewatching'].astype('boolean')

print(f'\nis_rewatching dtype : {df_rt["is_rewatching"].dtype}')
print('Valori dopo la conversione:')
print(df_rt['is_rewatching'].value_counts(dropna=False))

Valori is_rewatching prima della conversione:
is_rewatching
0.0    119457481
NaN      3775028
1.0       102796
Name: count, dtype: int64

is_rewatching dtype : boolean
Valori dopo la conversione:
is_rewatching
False    119457481
<NA>       3775028
True        102796
Name: count, dtype: Int64


### 2.6 `num_watched_episodes`

Questa colonna indica quanti episodi dell'anime l'utente ha guardato al momento del salvataggio dei dati.

Usiamo `analyze` per l'ispezione su eventuali valori nulli o anomali. 

In [60]:
analyze(df_rt['num_watched_episodes'])


  Nome serie                     num_watched_episodes
  dtype                          int64
  Dimensione                     123,335,305
  Range indice                   0 … 123351960
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   123,335,305
  Valori non nulli               123,335,305  (100.00%)
  Null / NaN                     0  (0.00%)
  Zeri                           35,641,162  (28.90%)
  Positivi                       87,694,143   (71.10%)
  Negativi                       0   (0.00%)
  Valori unici                   1,912  (0.00%)

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            0
  Max                            65,535
  Range                          65,535
  Media                          12.6593
  Mediana                        4.0000
  Moda/e              

**Osservazioni:**  

- Nessun null, dtype già `int64`. 
- Il valore `0` è atteso per entry con `status = 'plan_to_watch'` o per anime non ancora iniziati. 
- Nella distribuzione si nota un conteggio anomalo per l'intervallo 62,258 – 65,535. Si tratta di un comportamento di MAL dove i numeri vengono salvati come interi a 16 bit che permettono di sappresentare al massimo 65,536 valori. Essendo una percentuale molto bassa, non influiscono l'analisi e vanno tenute. 

Nessuna pulizia necessaria. 

### 2.7 Chiave composita `(username, anime_id)`

La combinazione `(username, anime_id)` identifica univocamente una riga: ogni utente ha al più un entry per anime.

In [61]:
n_pk_dup = df_rt.duplicated(subset=['username', 'anime_id'], keep=False).sum()
print(f'Duplicati su (username, anime_id) sull\'intero dataset: {n_pk_dup:,}')

if n_pk_dup == 0:
    print('→ Chiave composita univoca, nessuna operazione richiesta.')
else:
    print('→ Presenza di duplicati: rimozione necessaria.')
    df_rt = df_rt[~df_rt.duplicated(subset=['username', 'anime_id'], keep='first')].reset_index(drop=True)
    print(f'Righe dopo rimozione: {len(df_rt):,}')

Duplicati su (username, anime_id) sull'intero dataset: 12
→ Presenza di duplicati: rimozione necessaria.
Righe dopo rimozione: 123,335,299


Sono stati trovati 12 duplicati che abbiamo rimosso. La chiave composita `(username, anime_id)` adesso è univoca sull'intero dataset. Il dataset è pronto per il salvataggio.

## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effettuiamo il salvataggio del dataset finale.

In [62]:
print('Riepilogo Dataset Pulito')
print(f'Righe originali      : {n_originale:>15,}')
print(f'Righe dopo cleaning  : {len(df_rt):>15,}')
print(f'Righe rimosse totali : {n_originale - len(df_rt):>15,}')
print()
df_rt.to_csv('../datasets_cleaned/ratings_clean.csv', index=False)
print('Salvato: datasets_cleaned/ratings_clean.csv')

Riepilogo Dataset Pulito
Righe originali      :     124,298,357
Righe dopo cleaning  :     123,335,299
Righe rimosse totali :         963,058

Salvato: datasets_cleaned/ratings_clean.csv
